In [1]:
%load_ext autoreload
%autoreload 2

import time
import pandas as pd

from eventflow.config import EXPERIMENT_LOG, TUNING_FOLD
from eventflow.data import load_train, make_train_examples
from eventflow.lgbm import LightGBMModel
from eventflow.validation import add_folds, cross_validate

train = load_train()
examples = add_folds(make_train_examples(train))
print(f"{len(examples):,} practice questions from {examples['sequence_id'].nunique():,} machines")

596,953 practice questions from 6,500 machines


In [2]:
# Tune on fold 0 only. Each run learns from ALL training questions of folds 1-4 (90% of those
# machines; the other 10% are held back for early stopping), not the 150,000 sample of the quick check.
# The first run also makes the features (about a minute); the later runs reuse them.
candidates = [
    {"num_leaves": 63},                              # the quick-check setting
    {"num_leaves": 31},                              # smaller trees
    {"num_leaves": 127},                             # bigger trees
    {"num_leaves": 63, "min_child_samples": 200},    # every leaf needs at least 200 examples (smoother) Every leaf must cover at least 200 questions, so the rules are based on more evidence. That gives smoother predictions and less memorising.
]
rows = []
for settings in candidates:
    start = time.time()
    result = cross_validate(lambda: LightGBMModel(**settings), train, examples, folds=[TUNING_FOLD])
    rows.append({**settings, "logloss": result.mean, "mimic": result.mimic,
                 "minutes": round((time.time() - start) / 60, 1)})
tuning = pd.DataFrame(rows)
best = candidates[tuning["logloss"].idxmin()]
print("Best settings:", best)
tuning.round(4)

  stopped at round 198
fold 0: logloss 1.4820  mimic 1.4810  (152.6 s)
  stopped at round 256
fold 0: logloss 1.4809  mimic 1.4812  (130.4 s)
  stopped at round 142
fold 0: logloss 1.4860  mimic 1.4803  (163.1 s)
  stopped at round 184
fold 0: logloss 1.4788  mimic 1.4795  (155.9 s)
Best settings: {'num_leaves': 63, 'min_child_samples': 200}


,num_leaves,logloss,mimic,minutes,min_child_samples
0,63,1.4820,1.4810,2.5,NaN
1,31,1.4809,1.4812,2.2,NaN
2,127,1.4860,1.4803,2.7,NaN
3,63,1.4788,1.4795,2.6,200.0


In [3]:
# The best settings on all 5 folds. Out-of-fold predictions are saved to outputs/oof/lgbm_all.parquet
# for the final comparison and the error analysis.
result = cross_validate(lambda: LightGBMModel(**best), train, examples, name="lgbm_all")
result

  stopped at round 184
fold 0: logloss 1.4788  mimic 1.4795  (159.1 s)
  stopped at round 172
fold 1: logloss 1.4799  mimic 1.4260  (144.8 s)
  stopped at round 172
fold 2: logloss 1.4832  mimic 1.5233  (142.2 s)
  stopped at round 195
fold 3: logloss 1.4842  mimic 1.4593  (164.5 s)
  stopped at round 201
fold 4: logloss 1.4785  mimic 1.4843  (164.6 s)


CVResult(lgbm_all: folds 0-4, logloss 1.4809 ± 0.0026, mimic 1.4745)

In [4]:
# Log to experiments/results.csv. Run this cell ONCE (re-running adds a duplicate row).
result.log(run_id="lgbm_all", author="Nam", model="LightGBM", feature_set="all 6 groups (110 features)",
           params={**LightGBMModel(**best).params, "patience": 50},
           notes="5-fold cross_validate; early stopping on 10% held-back training machines")
print(pd.read_csv(EXPERIMENT_LOG).tail(1).T)

                                                              6
timestamp                                   2026-11-28T17:15:13
run_id                                                 lgbm_all
author                                                      Nam
model                                                  LightGBM
feature_set                         all 6 groups (110 features)
folds                                                       0-4
val_logloss                                              1.4809
val_std                                                  0.0026
val_mimic                                               1.47448
val_folds          [1.47878, 1.47986, 1.4832, 1.48416, 1.47853]
test_logloss                                                NaN
params        {"learning_rate": 0.05, "min_child_samples": 2...
notes         5-fold cross_validate; early stopping on 10% h...


In [5]:
# Ablation: leave ONE feature group out at a time (same settings, all 5 folds) and compare with the
# full model fold by fold. The more the log loss goes UP without a group, the more that group added
# on top of all the others. About 12 minutes per group (features are reused from the cache).
all_groups = ["recent", "counts", "since", "habits", "timing", "sensors"]
ablation = {}
for left_out in all_groups:
    groups = [g for g in all_groups if g != left_out]
    ablation[left_out] = cross_validate(lambda: LightGBMModel(groups=groups, print_round=False, **best),
                                        train, examples, verbose=False)
    print(f"without {left_out:8s}", ablation[left_out])

without recent   CVResult(unnamed: folds 0-4, logloss 1.4810 ± 0.0025, mimic 1.4725)
without counts   CVResult(unnamed: folds 0-4, logloss 1.5015 ± 0.0028, mimic 1.4944)
without since    CVResult(unnamed: folds 0-4, logloss 1.5116 ± 0.0033, mimic 1.5075)
without habits   CVResult(unnamed: folds 0-4, logloss 1.4801 ± 0.0024, mimic 1.4739)
without timing   CVResult(unnamed: folds 0-4, logloss 1.4892 ± 0.0026, mimic 1.4790)
without sensors  CVResult(unnamed: folds 0-4, logloss 1.4893 ± 0.0027, mimic 1.4799)


run 6 time, each time only use 5 feature group to check which feature group contribute the most

In [6]:
# Compare with the full model (`result`, Cell 3), fold by fold: change = ablated - full.
# build the table to display the result from the prev cell
rows = []
for left_out, r in ablation.items():
    change = r.per_fold["logloss"] - result.per_fold["logloss"]      # one number per fold
    rows.append({"left out": left_out, "logloss": r.mean, "change": change.mean(),
                 "worse on": f"{(change > 0).sum()} of 5 folds", "mimic": r.mimic})
ablation_table = pd.DataFrame(rows).sort_values("change", ascending=False)
ablation_table.round(4)

,left out,logloss,change,worse on,mimic
2,since,1.5116,0.0307,5 of 5 folds,1.5075
1,counts,1.5015,0.0206,5 of 5 folds,1.4944
5,sensors,1.4893,0.0084,5 of 5 folds,1.4799
4,timing,1.4892,0.0083,5 of 5 folds,1.4790
0,recent,1.4810,0.0001,4 of 5 folds,1.4725
3,habits,1.4801,-0.0008,0 of 5 folds,1.4739


In [7]:
# Log each ablation run to experiments/results.csv. Run this cell ONCE.
for left_out, r in ablation.items():
    r.log(run_id=f"lgbm_without_{left_out}", author="Nam", model="LightGBM",
          feature_set=f"all groups except {left_out}",
          params={**LightGBMModel(**best).params, "patience": 50},
          notes="ablation: one feature group left out, 5-fold cross_validate")
print(pd.read_csv(EXPERIMENT_LOG).tail(len(ablation))[["run_id", "val_logloss", "val_std", "val_mimic"]])

                  run_id  val_logloss  val_std  val_mimic
7    lgbm_without_recent      1.48099  0.00253    1.47254
8    lgbm_without_counts      1.50146  0.00275    1.49437
9     lgbm_without_since      1.51157  0.00331    1.50754
10   lgbm_without_habits      1.48012  0.00240    1.47387
11   lgbm_without_timing      1.48919  0.00256    1.47903
12  lgbm_without_sensors      1.48931  0.00266    1.47992


### What the ablation shows

Full model (all 6 groups, 110 features): logloss **1.4809 ± 0.0026**. "change" = logloss without the group − full model, compared fold by fold.

- **since** (+0.031) and **counts** (+0.021) matter most: without either one, the model is worse on all 5 folds.
- **sensors** (+0.008) and **timing** (+0.008) help less, but still on every fold.
- **recent** (+0.0001) adds nothing once *since* is in. `since_X = 0` marks the current event, so the last 3 events and the run length can be read from the since columns.
- **habits** (−0.0008) is slightly better left out (worse on 0 of 5 folds). *counts* already gives the recent event mix. It was already borderline in the quick check (06): 1.5185 → 1.5165, with mimic getting worse.

An ablation measures what a group adds **on top of the others**, not how useful it is alone. *recent* is strong alone: the last-3 table baseline reaches 1.644 on it.

Without recent and habits **together**: logloss **1.4801 ± 0.0026** (change −0.0008, worse on 0 of 5 folds; mimic 1.4688 vs 1.4745). That is the **same score** as all 6 groups (the difference is much smaller than the 0.0026 spread between folds), with 87 instead of 110 features and about 25% faster per fold. So these 4 groups are enough: **`lgbm_4groups` is the final LightGBM** (out-of-fold predictions in `outputs/oof/lgbm_4groups.parquet`).

In [9]:
# Leave out recent AND habits together (each looked redundant on its own).
# Same settings as the full model; out-of-fold predictions saved as lgbm_4groups.
groups4 = ["counts", "since", "timing", "sensors"]
result4 = cross_validate(lambda: LightGBMModel(groups=groups4, **best), train, examples,
                         name="lgbm_4groups")

# Compare with the full model fold by fold, same as the ablation table
change4 = result4.per_fold["logloss"] - result.per_fold["logloss"]
print(f"change vs all 6 groups: {change4.mean():+.4f}, worse on {(change4 > 0).sum()} of 5 folds")
result4

  stopped at round 197
fold 0: logloss 1.4783  mimic 1.4746  (119.3 s)
  stopped at round 190
fold 1: logloss 1.4791  mimic 1.4163  (109.5 s)
  stopped at round 205
fold 2: logloss 1.4824  mimic 1.5211  (112.5 s)
  stopped at round 192
fold 3: logloss 1.4831  mimic 1.4589  (110.9 s)
  stopped at round 180
fold 4: logloss 1.4774  mimic 1.4729  (114.3 s)
change vs all 6 groups: -0.0008, worse on 0 of 5 folds


CVResult(lgbm_4groups: folds 0-4, logloss 1.4801 ± 0.0026, mimic 1.4688)

In [10]:
# Log the 4-group run to experiments/results.csv. Run this cell ONCE.
result4.log(run_id="lgbm_4groups", author="Nam", model="LightGBM",
            feature_set="counts + since + timing + sensors (87 features)",
            params={**LightGBMModel(**best).params, "patience": 50, "groups": groups4},
            notes="recent and habits left out together, 5-fold cross_validate")
print(pd.read_csv(EXPERIMENT_LOG).tail(1)[["run_id", "val_logloss", "val_std", "val_mimic"]])

          run_id  val_logloss  val_std  val_mimic
13  lgbm_4groups      1.48006  0.00255    1.46876
